# SmartPark Maroc - environment check

Run all cells (`Run > Run All Cells`) to verify your local setup.
This notebook does **not** need a dataset or model weights.

Before starting, make sure the correct environment is active:

```bash
conda activate smartpark
jupyter lab notebooks/01_environment_check.ipynb
```

In [ ]:
import sys

import cv2
import matplotlib
import numpy as np
import pandas as pd
import torch
import ultralytics
from PIL import __version__ as pillow_version

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("OpenCV      :", cv2.__version__)
print("Ultralytics :", ultralytics.__version__)
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Matplotlib  :", matplotlib.__version__)
print("Pillow      :", pillow_version)

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA build    :", torch.version.cuda)

assert not torch.cuda.is_available(), "This project must run on CPU only"
print("CPU-only setup OK")

## Smoke test: image processing utilities

The next cell creates a synthetic image, saves it to `data/processed/`
and displays it. No personal data or external dataset is used.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Find the project root whether this notebook is run from notebooks/ or the root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.processing.image_utils import (  # noqa: E402
    bgr_to_rgb,
    draw_boxes,
    resize_image,
    save_image,
    to_grayscale,
)

image = np.full((240, 480, 3), 245, dtype=np.uint8)
cv2.rectangle(image, (150, 80), (330, 160), (0, 120, 255), -1)
cv2.putText(image, "12345-A-6", (165, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (20, 20, 20), 2)

boxed = draw_boxes(image, [(140, 70, 340, 170)], color=(0, 200, 0), thickness=3)
output_path = save_image(PROJECT_ROOT / "data" / "processed" / "smoke_test.png", boxed)
small = resize_image(boxed, width=240)
gray = to_grayscale(small)

print("Saved:", output_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(bgr_to_rgb(small))
axes[0].set_title("BGR -> RGB")
axes[0].axis("off")
axes[1].imshow(gray, cmap="gray")
axes[1].set_title("Grayscale")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Next steps

1. Put vehicle photos in `data/raw/` (do not commit them).
2. Place trained YOLO weights at `models/plate_detector.pt`.
3. Launch the demo with `streamlit run app/app.py`.
4. Plug an OCR engine into `src/recognition/ocr.py`.